# Can we predict 30-day readmission?
Target: `readmit_30d`. Follows `checklist.md`.

Design notes
- `person_key` repeats (several stays per person), so every split holds **whole persons** (checklist 1, 2).
- There is no calendar date, so no "later rows" split is possible; `visit_number` is kept as a feature (known at the time of the stay).
- Features are all recorded at admission or discharge, i.e. known when we would predict readmission after discharge. `person_key` is an identifier and is dropped (checklist 7).

In [ ]:
import numpy as np, pandas as pd, json
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support, brier_score_loss
df=pd.read_csv('data.csv')
print(df.shape, df.person_key.nunique(), 'persons; prevalence', df.readmit_30d.mean().round(3))

## 1. Hold out the test set (whole persons) before any modelling

In [ ]:
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=0)
tr_idx,te_idx=next(gss.split(df,groups=df.person_key))
train,test=df.iloc[tr_idx].reset_index(drop=True),df.iloc[te_idx].reset_index(drop=True)
assert not set(train.person_key)&set(test.person_key)
print(len(train),len(test),train.readmit_30d.mean().round(3),test.readmit_30d.mean().round(3))
FEATS=['visit_number','birth_year','sex','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
Xtr,ytr,gtr=train[FEATS],train.readmit_30d,train.person_key
Xte,yte=test[FEATS],test.readmit_30d

## 2. Candidate models (named before any scoring)
1. `baseline_prior` – predicts the training prevalence (trivial).
2. `logreg_C{0.01,0.1,1}` – scaled numerics, one-hot sex and postcode area, L2 logistic regression.
3. `hgb_{shallow,deep}` – histogram gradient boosting (depth 2 / lr 0.05, depth 4 / lr 0.05), 150 iterations.

Selection metric: mean ROC AUC over 5 person-grouped CV folds on the training rows (PR AUC reported too). Threshold: the one maximising F1 on pooled out-of-fold predictions of the chosen model. All preprocessing lives inside the pipelines, so it is fit within each fold (checklist 3).

In [ ]:
num=['visit_number','birth_year','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
cat=['sex','postcode_area']
def prep():
    return ColumnTransformer([('n',Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())]),num),
                              ('c',OneHotEncoder(handle_unknown='ignore'),cat)])
def hgb(**k):
    return Pipeline([('p',ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(handle_unknown='ignore'),['sex'])])),
                     ('m',HistGradientBoostingClassifier(max_iter=150,random_state=0,**k))])
cands={'baseline_prior':Pipeline([('m',DummyClassifier(strategy='prior'))]),
 **{f'logreg_C{C}':Pipeline([('p',prep()),('m',LogisticRegression(C=C,max_iter=2000))]) for C in (0.01,0.1,1)},
 'hgb_shallow':hgb(max_depth=2,learning_rate=0.05),'hgb_deep':hgb(max_depth=4,learning_rate=0.05)}
print(list(cands))

## 3. Cross-validation on training rows only (GroupKFold by person)

In [ ]:
cv=GroupKFold(n_splits=5)
folds=list(cv.split(Xtr,ytr,gtr))
oof={};rows=[]
for name,m in cands.items():
    p=cross_val_predict(m,Xtr,ytr,cv=folds,method='predict_proba')[:,1]
    oof[name]=p
    fa=[roc_auc_score(ytr.iloc[v],p[v]) for _,v in folds]
    rows.append((name,np.mean(fa),np.std(fa),average_precision_score(ytr,p),brier_score_loss(ytr,p)))
cvres=pd.DataFrame(rows,columns=['model','cv_auc','auc_sd','oof_pr_auc','brier']).set_index('model').round(4)
cvres

## 4. Choose model and threshold on CV only, and record the choice

In [ ]:
best=cvres.cv_auc.idxmax()
p=oof[best]
ths=np.linspace(0.05,0.6,111)
f1=[precision_recall_fscore_support(ytr,p>=t,average='binary',zero_division=0)[2] for t in ths]
thr=float(ths[int(np.argmax(f1))])
choice={'model':best,'features':FEATS,'threshold':round(thr,3)}
print('CHOICE (fixed before test scoring):',json.dumps(choice,indent=1))

**Recorded choice:** see the printed `CHOICE` above — the model with the best CV AUC, its F1-optimal CV threshold. Nothing below changes it.

## 5. Refit on all training rows; score the test set once (headline)

In [ ]:
final=cands[best].fit(Xtr,ytr)
pt=final.predict_proba(Xte)[:,1]
auc=roc_auc_score(yte,pt); ap=average_precision_score(yte,pt)
pr,rc,f,_=precision_recall_fscore_support(yte,pt>=thr,average='binary',zero_division=0)
rng=np.random.default_rng(0); pers=test.person_key.values; up=np.unique(pers)
idx={u:np.where(pers==u)[0] for u in up}; bs=[]
for _ in range(1000):
    s=np.concatenate([idx[u] for u in rng.choice(up,len(up))])
    bs.append(roc_auc_score(yte.values[s],pt[s]))
print(f'TEST ({best}): AUC {auc:.3f} (95% CI by person bootstrap {np.percentile(bs,2.5):.3f}-{np.percentile(bs,97.5):.3f}), PR-AUC {ap:.3f} (prevalence {yte.mean():.3f})')
print(f'at threshold {thr:.2f}: precision {pr:.3f} recall {rc:.3f} F1 {f:.3f}')

## 6. Reading the result
Compare the test AUC and its interval with 0.5 (chance) and with the baseline's CV AUC in the table above; PR-AUC should be read against the prevalence. Persons in this dataset are a single sample, and no calendar information exists, so temporal generalisation is untested.